<a href="https://colab.research.google.com/github/latikasekhri6577/Langchain-LangGraph-Course/blob/main/Multi_UserRAGbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain langchain-community langchain-text-splitters langchain-core langchain-google-genai langchain-chroma pypdf langsmith

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.7/401.7 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.0/268.0 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 M

In [13]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GOOGLE_API_KEY"] = userdata.get("google_key")

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

print(model.invoke("Say hello in one line.").content)

Hello!


In [9]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

PDF_PATH = "/Keys in DBMS.pdf"   # change this to your file's name

docs = PyPDFLoader(PDF_PATH).load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(docs)

print(len(docs), "pages ->", len(chunks), "chunks")
print(chunks[0].page_content[:300])

9 pages -> 15 chunks
What are Keys in DBMS? 
KEYS in DBMS is an attribute or set of attributes which helps you to identify a row(tuple) in a relation(table). They 
allow you to find the relation between two tables. Keys help you uniquely identify a row in a table by a combination 
of one or more columns in that table. K


In [10]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

vectorstore = Chroma.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("Stored chunks:", vectorstore._collection.count())

Stored chunks: 15


In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

prompt = ChatPromptTemplate.from_template(
    "Answer only from the context. If the answer is not there, say you don't know.\n\n"
    "Context:\n{context}\n\nQuestion: {question}"
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt | model | StrOutputParser()
)

In [12]:
print(rag_chain.invoke("What is a primary key?"))

PRIMARY KEY in DBMS is a column or group of columns in a table that uniquely identify every row in that table. The Primary Key can’t be a duplicate meaning the same value can’t appear more than once in the table. A table cannot have more than one primary key.


In [14]:
from langchain_core.messages import HumanMessage, AIMessage

histories = {}   # user_id -> that user's list of messages

def get_history(user_id):
    if user_id not in histories:
        histories[user_id] = []
    return histories[user_id]

def add_turn(user_id, question, answer):
    h = get_history(user_id)
    h.append(HumanMessage(content=question))
    h.append(AIMessage(content=answer))

# test with fake data (no LLM calls)
add_turn("latika", "What is a primary key?", "A unique identifier for a row.")
add_turn("friend", "What is a foreign key?", "A column that refers to another table's primary key.")

print("latika:", len(get_history("latika")), "messages")
print("friend:", len(get_history("friend")), "messages")
print(get_history("latika")[0].content)
print(get_history("friend")[0].content)

latika: 2 messages
friend: 2 messages
What is a primary key?
What is a foreign key?


In [15]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", "Rewrite the latest question so it makes sense without the chat history. "
               "Do NOT answer it. If it is already standalone, return it unchanged."),
    MessagesPlaceholder("chat_history"),
    ("human", "{question}"),
])
rewrite_chain = rewrite_prompt | model | StrOutputParser()

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer only from the context. If the answer is not there, say you don't know.\n\n"
               "Context:\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{question}"),
])
qa_chain = qa_prompt | model | StrOutputParser()


def ask(user_id, question):
    history = get_history(user_id)[-6:]          # only THIS user's recent messages

    if history:                                   # rewrite follow-ups
        standalone = rewrite_chain.invoke(
            {"question": question, "chat_history": history}
        ).strip() or question
    else:
        standalone = question

    docs = retriever.invoke(standalone)           # find chunks
    answer = qa_chain.invoke({
        "context": format_docs(docs),
        "question": question,
        "chat_history": history,
    })

    add_turn(user_id, question, answer)           # save to THIS user only
    return answer

In [25]:
# clear the fake test data from Step 5a first
histories.clear()

print(ask("latika", "What is a primary key?"))
print("-----")


PRIMARY KEY in DBMS is a column or group of columns in a table that uniquely identify every row in that table. The Primary Key can’t be a duplicate, meaning the same value can’t appear more than once in the table. A table cannot have more than one primary key.
-----


In [18]:
print(ask("latika", "How is it different from a foreign key?"))
# print("-----")
# print(ask("friend", "What is a foreign key?"))

Here are the differences between a primary key and a foreign key:

*   **Purpose:**
    *   **Primary Key:** Helps you to uniquely identify a record in the table.
    *   **Foreign Key:** It is a field in the table that is the primary key of another table.
*   **Null Values:**
    *   **Primary Key:** Never accepts null values.
    *   **Foreign Key:** May accept multiple null values.
*   **Indexing:**
    *   **Primary Key:** Is a clustered index, and data in the DBMS table are physically organized in the sequence of the clustered index.
    *   **Foreign Key:** Cannot automatically create an index, clustered or non-clustered. However, you can manually create an index on the foreign key.


In [19]:
print(ask("friend", "What is a foreign key?"))

FOREIGN KEY is a column that creates a relationship between two tables. The purpose of Foreign keys is to maintain data integrity and allow navigation between two different instances of an entity. It acts as a cross-reference between two tables as it references the primary key of another table.


In [20]:
print(len(get_history("latika")), len(get_history("friend")))

4 2


In [21]:
import os
from google.colab import userdata
from langsmith import traceable

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
os.environ["LANGSMITH_PROJECT"] = "multi-user-rag"

ask_traced = traceable(name="multi_user_rag")(ask)

def chat(user_id, question):
    return ask_traced(
        user_id, question,
        langsmith_extra={"metadata": {"user_id": user_id}, "tags": ["multi-user"]},
    )

In [22]:
print(chat("latika", "What is a candidate key?"))

CANDIDATE KEY in SQL is a set of attributes that uniquely identify tuples in a table. Candidate Key is a super key with no repeated attributes. The Primary key should be selected from the candidate keys. Every table must have at least a single candidate key. A table can have multiple candidate keys but only a single primary key.


In [23]:
import os
print(os.environ.get("LANGSMITH_TRACING"), os.environ.get("LANGSMITH_PROJECT"), bool(os.environ.get("LANGSMITH_API_KEY")))

true multi-user-rag True


In [24]:
import langsmith as ls

with ls.tracing_context(enabled=True, project_name="multi-user-rag"):
    print(chat("latika", "What is a super key?"))

A superkey is a group of single or multiple keys which identifies rows in a table. A Super key may have additional attributes that are not needed for unique identification.
